# Cross-Validation — Multi-Source Dataset (Proposed Model Only) — PyTorch

In [1]:
import os, copy, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import chi2, wilcoxon, norm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, classification_report)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import label_binarize

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.max_open_warning': 0})

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if DEVICE.type == 'cuda': torch.backends.cudnn.benchmark = True

print(f'PyTorch     : {torch.__version__}')
print(f'Device      : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU         : {torch.cuda.get_device_name(0)}')
    print(f'VRAM        : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')


PyTorch     : 2.5.1+cu121
Device      : cuda
GPU         : NVIDIA RTX 5000 Ada Generation
VRAM        : 32.0 GB


In [2]:
BATCH_SIZE  = 64
IMAGE_SIZE  = (224, 224)
EPOCHS      = 50
NUM_CLASSES = 6
RANDOM_SEED = 42
DATA_DIR    = r'../Rice_Disease_Dataset'

OUTPUT_DIR = 'CV_Results'
os.makedirs(f'{OUTPUT_DIR}/Histories',  exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/Plots',      exist_ok=True)
os.makedirs(f'{OUTPUT_DIR}/BestModels', exist_ok=True)

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

def load_filepaths_labels(directory):
    filepaths, labels = [], []
    for cls in sorted(os.listdir(directory)):
        cls_dir = os.path.join(directory, cls)
        if not os.path.isdir(cls_dir): continue
        for f in os.listdir(cls_dir):
            if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp')):
                filepaths.append(os.path.join(cls_dir, f)); labels.append(cls)
    return filepaths, labels

fp, lb = load_filepaths_labels(DATA_DIR)
full_df = pd.DataFrame({'filepath': fp, 'label': lb})

df_cv, df_test = train_test_split(full_df, test_size=0.20, stratify=full_df['label'], random_state=RANDOM_SEED)
class_names   = sorted(full_df['label'].unique())
class_to_idx  = {c: i for i, c in enumerate(class_names)}
NUM_CLASSES   = len(class_names)

train_labels_int     = [class_to_idx[l] for l in df_cv['label']]
class_weights_arr    = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=train_labels_int)
class_weights_tensor = torch.tensor(class_weights_arr, dtype=torch.float32).to(DEVICE)

print(f"Total Images: {len(full_df)} | CV Set: {len(df_cv)} | Test Set: {len(df_test)}")
print(f"Classes: {class_names}")


Total Images: 14758 | CV Set: 11806 | Test Set: 2952
Classes: ['bacterial_leaf_blight', 'brown_spot', 'healthy', 'leaf_blast', 'leaf_scaled', 'narrow_brown_spot']


In [3]:
IDENTITY_NORM = transforms.Normalize([0.0, 0.0, 0.0], [1.0, 1.0, 1.0])
IMAGENET_NORM = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
MINUSONE_NORM = transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])

class ImageDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df       = df.reset_index(drop=True)
        self.transform = transform
        self.encoded  = [class_to_idx[l] for l in self.df['label']]
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        img = Image.open(self.df.iloc[idx]['filepath']).convert('RGB')
        return self.transform(img) if self.transform else img, self.encoded[idx]

def make_loaders(df_tr, df_vl, train_t, val_t, batch_size=BATCH_SIZE):
    pin = DEVICE.type == 'cuda'
    tr  = DataLoader(ImageDataset(df_tr, train_t), batch_size=batch_size, shuffle=True,
                     num_workers=0, pin_memory=pin, drop_last=True)
    vl  = DataLoader(ImageDataset(df_vl, val_t),  batch_size=batch_size, shuffle=False,
                     num_workers=0, pin_memory=pin)
    return tr, vl

def apply_freeze_like_keras(backbone, n_trainable=35, label="backbone"):
    for p in backbone.parameters(): p.requires_grad = True
    leaf_modules = [(n, m) for n, m in backbone.named_modules() if len(list(m.children())) == 0]
    n_total  = len(leaf_modules)
    n_freeze = max(0, n_total - n_trainable)
    for _, mod in leaf_modules[:n_freeze]:
        for p in mod.parameters(recurse=False): p.requires_grad = False
    print(f"  {label}: {n_freeze} frozen | {n_total - n_freeze} trainable (last {n_trainable})")
    return n_total - n_freeze

class EarlyStopping:
    def __init__(self, patience=7):
        self.patience     = patience
        self.best_loss    = float('inf')
        self.counter      = 0
        self.best_weights   = None
        self.best_acc       = 0.0
        self.best_f1        = 0.0
        self.best_precision = 0.0
        self.best_recall    = 0.0

    def __call__(self, val_loss, val_acc, val_f1, val_precision, val_recall, model):
        if val_loss < self.best_loss:
            self.best_loss      = val_loss
            self.counter        = 0
            self.best_weights   = copy.deepcopy(model.state_dict())
            self.best_acc       = val_acc
            self.best_f1        = val_f1
            self.best_precision = val_precision
            self.best_recall    = val_recall
            return False
        self.counter += 1
        return self.counter >= self.patience

    def restore(self, model):
        if self.best_weights:
            model.load_state_dict(self.best_weights)


In [4]:
def run_kfold_cv(model_name, build_fn, train_transform, val_transform, df, n_folds=5):
    skf       = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=RANDOM_SEED)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor, label_smoothing=0.1)
    fold_results, all_histories = [], []
    safe = model_name.replace(' ', '_').replace('(', '').replace(')', '').replace('-', '')

    global_best_acc = -1.0
    best_model_path = f'{OUTPUT_DIR}/BestModels/{safe}_best.pth'

    print(f"\n{'='*80}\n  MODEL: {model_name} | {n_folds}-FOLD CV\n{'='*80}")

    for fold, (tr_idx, vl_idx) in enumerate(skf.split(df['filepath'], df['label']), 1):
        df_tr, df_vl     = df.iloc[tr_idx].reset_index(drop=True), df.iloc[vl_idx].reset_index(drop=True)
        tr_loader, vl_loader = make_loaders(df_tr, df_vl, train_transform, val_transform)

        model     = build_fn(NUM_CLASSES).to(DEVICE)
        optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-5)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)
        scaler    = torch.cuda.amp.GradScaler(enabled=(DEVICE.type == 'cuda'))
        es        = EarlyStopping(patience=7)
        fold_history = {'loss': [], 'accuracy': [], 'precision': [], 'recall': [], 'f1_score': [],
                        'val_loss': [], 'val_accuracy': [], 'val_precision': [], 'val_recall': [], 'val_f1_score': []}

        print(f"\nFold {fold}/{n_folds}")
        n_steps = len(tr_loader)

        for epoch in range(1, EPOCHS + 1):
            t0 = time.time()
            model.train()
            tr_loss, tr_preds, tr_labs = 0.0, [], []

            for bi, (imgs, labs) in enumerate(tr_loader, 1):
                imgs, labs = imgs.to(DEVICE), labs.to(DEVICE)
                optimizer.zero_grad(set_to_none=True)
                with torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):
                    out  = model(imgs)
                    if isinstance(out, torchvision.models.inception.InceptionOutputs): out = out.logits
                    loss = criterion(out, labs)
                scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()

                tr_loss += loss.item() * imgs.size(0)
                tr_preds.extend(out.argmax(1).cpu().numpy())
                tr_labs.extend(labs.cpu().numpy())

                elapsed_b = time.time() - t0
                filled = int(30 * bi / n_steps)
                bar = '=' * filled + '>' * (1 if filled < 30 else 0) + '.' * (30 - filled - (1 if filled < 30 else 0))
                print(f"\r  Epoch {epoch}/{EPOCHS} [{bar}] {bi}/{n_steps} - {elapsed_b:.0f}s", end='', flush=True)

            model.eval()
            vl_loss, vl_preds, vl_labs = 0.0, [], []
            with torch.no_grad():
                for imgs, labs in vl_loader:
                    imgs, labs = imgs.to(DEVICE), labs.to(DEVICE)
                    with torch.cuda.amp.autocast(enabled=(DEVICE.type == 'cuda')):
                        out = model(imgs)
                        if isinstance(out, torchvision.models.inception.InceptionOutputs): out = out.logits
                        vl_loss += criterion(out, labs).item() * imgs.size(0)
                    vl_preds.extend(out.argmax(1).cpu().numpy())
                    vl_labs.extend(labs.cpu().numpy())

            tr_m = {'loss':      tr_loss / len(tr_labs),
                    'accuracy':  accuracy_score(tr_labs, tr_preds),
                    'precision': precision_score(tr_labs, tr_preds, average='weighted', zero_division=0),
                    'recall':    recall_score(tr_labs, tr_preds, average='weighted', zero_division=0),
                    'f1_score':  f1_score(tr_labs, tr_preds, average='weighted')}
            vl_m = {'val_loss':      vl_loss / len(vl_labs),
                    'val_accuracy':  accuracy_score(vl_labs, vl_preds),
                    'val_precision': precision_score(vl_labs, vl_preds, average='weighted', zero_division=0),
                    'val_recall':    recall_score(vl_labs, vl_preds, average='weighted', zero_division=0),
                    'val_f1_score':  f1_score(vl_labs, vl_preds, average='weighted')}

            for k in fold_history:
                src = tr_m if k not in vl_m else vl_m
                fold_history[k].append(round(src[k], 4))
            scheduler.step(vl_m['val_loss'])

            elapsed_e = time.time() - t0
            print(f"\n  Epoch {epoch}/{EPOCHS} - {elapsed_e:.0f}s - loss: {tr_m['loss']:.4f} - acc: {tr_m['accuracy']:.4f} - val_loss: {vl_m['val_loss']:.4f} - val_acc: {vl_m['val_accuracy']:.4f}")

            if es(vl_m['val_loss'], vl_m['val_accuracy'], vl_m['val_f1_score'], vl_m['val_precision'], vl_m['val_recall'], model):
                print(f"  Early Stopping at epoch {epoch} (best val_loss={es.best_loss:.4f})")
                break

        es.restore(model)
        best_acc = es.best_acc
        best_f1  = es.best_f1
        print(f"\nFold {fold} Complete | Acc: {best_acc:.4f} | Prec: {es.best_precision:.4f} | Rec: {es.best_recall:.4f} | F1: {best_f1:.4f} | Epochs: {epoch}")

        if best_acc > global_best_acc:
            global_best_acc = best_acc
            torch.save(model.state_dict(), best_model_path)
            print(f"  → New best fold checkpoint saved (acc={best_acc:.4f})")

        fold_results.append({'fold': fold, 'accuracy': best_acc, 'precision': es.best_precision, 'recall': es.best_recall, 'f1_score': best_f1})
        all_histories.append(fold_history)
        del model; torch.cuda.empty_cache()

    return pd.DataFrame(fold_results), all_histories, best_model_path


In [ ]:
# ── Shared TL Head ────────────────────────────────────────────────────────────
def make_tl_head(in_ch, nc):
    return nn.Sequential(
        nn.Conv2d(in_ch, in_ch, 3, padding=1, groups=in_ch), 
        nn.ReLU(True),
        nn.Conv2d(in_ch, 128, 1), 
        nn.ReLU(True), 
        nn.BatchNorm2d(128),
        nn.AdaptiveAvgPool2d(1), 
        nn.Flatten(), 
        nn.BatchNorm1d(128),
        nn.Linear(128, 128), 
        nn.ReLU(True), 
        nn.Dropout(0.4), 
        nn.Linear(128, nc)
    )

# ── Proposed Model (MobileNetV2 + custom TL head) ─────────────────────────────
class ProposedModel(nn.Module):
    def __init__(self, nc=NUM_CLASSES):
        super().__init__()
        self.backbone = torchvision.models.mobilenet_v2(
            weights=torchvision.models.MobileNet_V2_Weights.IMAGENET1K_V1).features
        apply_freeze_like_keras(self.backbone, n_trainable=35, label="MobileNetV2 backbone")
        self.head = make_tl_head(1280, nc)
    def forward(self, x):
        return self.head(self.backbone(x))

# ── Transforms ────────────────────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomRotation(25),
    transforms.RandomAffine(degrees=0, translate=(0.15, 0.15), shear=5.7),
    transforms.ToTensor(),
    MINUSONE_NORM
])
val_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    MINUSONE_NORM
])

# ── Model Registry (Proposed only) ───────────────────────────────────────────
MODELS = [
    ('Proposed', lambda nc: ProposedModel(nc), train_transform, val_transform)
]
print(f"Registered {len(MODELS)} model(s).")


Registered 1 model(s).


In [6]:
final_summaries  = {}
MODEL_BEST_PATHS = {}

for n_folds in [5, 10]:
    print(f"\n{'#'*80}\n  RUNNING {n_folds}-FOLD CROSS-VALIDATION\n{'#'*80}")
    for name, build_fn, t_t, v_t in MODELS:
        safe = name.replace(' ', '_').replace('(', '').replace(')', '').replace('-', '')

        fold_df, histories, best_path = run_kfold_cv(name, build_fn, t_t, v_t, df_cv, n_folds=n_folds)

        with open(f'{OUTPUT_DIR}/Histories/{safe}_{n_folds}fold.json', 'w') as f:
            json.dump(histories, f)
        fold_df.to_csv(f'{OUTPUT_DIR}/{safe}_{n_folds}fold.csv', index=False)

        mean_acc  = fold_df['accuracy'].mean()  * 100
        mean_prec = fold_df['precision'].mean() * 100
        mean_rec  = fold_df['recall'].mean()    * 100
        mean_f1   = fold_df['f1_score'].mean()  * 100
        final_summaries[f'{name}_{n_folds}Fold'] = {'Acc': mean_acc, 'Prec': mean_prec, 'Rec': mean_rec, 'F1': mean_f1}
        MODEL_BEST_PATHS[name] = best_path

        print(f"Saved | Acc: {mean_acc:.2f}% | Prec: {mean_prec:.2f}% | Rec: {mean_rec:.2f}% | F1: {mean_f1:.2f}%\n")

print("\n" + "="*60 + "\n  CROSS-VALIDATION SUMMARY\n" + "="*60)
summary_df = pd.DataFrame(final_summaries).T
print(summary_df.to_string())
summary_df.to_csv(f'{OUTPUT_DIR}/CV_Summary.csv')



################################################################################
  RUNNING 5-FOLD CROSS-VALIDATION
################################################################################

  MODEL: Proposed | 5-FOLD CV
  MobileNetV2 backbone: 104 frozen | 35 trainable (last 35)

Fold 1/5
  Epoch 1/50 [==============================] 147/147 - 30s
  Epoch 1/50 - 38s - loss: 1.7337 - acc: 0.3204 - val_loss: 1.6447 - val_acc: 0.3882
  Epoch 2/50 [==============================] 147/147 - 20s
  Epoch 2/50 - 25s - loss: 1.5484 - acc: 0.4904 - val_loss: 1.4943 - val_acc: 0.4941
  Epoch 3/50 [==============================] 147/147 - 21s
  Epoch 3/50 - 25s - loss: 1.4181 - acc: 0.5800 - val_loss: 1.3827 - val_acc: 0.5711
  Epoch 4/50 [==============================] 147/147 - 21s
  Epoch 4/50 - 25s - loss: 1.3176 - acc: 0.6230 - val_loss: 1.2970 - val_acc: 0.6164
  Epoch 5/50 [==============================] 147/147 - 20s
  Epoch 5/50 - 25s - loss: 1.2348 - acc: 0.6692 - val_loss: 1.

In [7]:
print("\n" + "="*80 + "\n  HELD-OUT TEST EVALUATION & STATISTICAL ANALYSIS\n" + "="*80)

test_preds = {}

for name, build_fn, _, val_t in MODELS:
    test_loader = DataLoader(
        ImageDataset(df_test, val_t),
        batch_size=BATCH_SIZE, shuffle=False, num_workers=0,
        pin_memory=(DEVICE.type == 'cuda')
    )
    model = build_fn(NUM_CLASSES).to(DEVICE)

    best_path = MODEL_BEST_PATHS.get(name)
    if best_path and os.path.exists(best_path):
        model.load_state_dict(torch.load(best_path, map_location=DEVICE))
        print(f"  Loaded checkpoint: {best_path}")
    else:
        print(f"  WARNING: No checkpoint found for {name}. Using random weights.")

    model.eval()
    yt, yp = [], []
    with torch.no_grad():
        for imgs, labs in test_loader:
            out = model(imgs.to(DEVICE))
            if isinstance(out, torchvision.models.inception.InceptionOutputs): out = out.logits
            yp.extend(out.argmax(1).cpu().numpy())
            yt.extend(labs.numpy())

    test_preds[name] = {'true': np.array(yt), 'pred': np.array(yp)}
    del model; torch.cuda.empty_cache()

# ── Statistical helpers ───────────────────────────────────────────────────────
def mcnemar_test(y_true, y_prop, y_base, base_name):
    b = int(np.sum((y_prop == y_true) & (y_base != y_true)))
    c = int(np.sum((y_prop != y_true) & (y_base == y_true)))
    chi2_stat = ((abs(b - c) - 1) ** 2) / (b + c) if (b + c) else 0
    p = 1 - chi2.cdf(chi2_stat, df=1)
    print(f"\nMcNemar: {base_name} vs Proposed | "
          f"chi2={chi2_stat:.4f}, p={p:.4f} | "
          f"{'Significant' if p < 0.05 else 'NS'}")
    return chi2_stat, p

def wilcoxon_test(name_prop, name_base, n_folds):
    safe_p = name_prop.replace(' ', '_').replace('(', '').replace(')', '').replace('-', '')
    safe_b = name_base.replace(' ', '_').replace('(', '').replace(')', '').replace('-', '')
    path_p = f'{OUTPUT_DIR}/{safe_p}_{n_folds}fold.csv'
    path_b = f'{OUTPUT_DIR}/{safe_b}_{n_folds}fold.csv'
    if not (os.path.exists(path_p) and os.path.exists(path_b)):
        print(f"  Wilcoxon skipped: missing CSV for {name_base} or {name_prop}")
        return None, None
    accs_p = pd.read_csv(path_p)['accuracy'].values
    accs_b = pd.read_csv(path_b)['accuracy'].values
    if len(accs_p) != len(accs_b):
        print("  Wilcoxon skipped: fold-count mismatch")
        return None, None
    stat, p = wilcoxon(accs_p, accs_b, alternative='greater')
    print(f"Wilcoxon ({n_folds}-fold): {name_base} vs Proposed | "
          f"W={stat:.4f}, p={p:.4f} | "
          f"{'Significant' if p < 0.05 else 'NS'}")
    return stat, p

def wilson_ci(y_true, y_pred, conf=0.95):
    n   = len(y_true)
    z   = norm.ppf(1 - (1 - conf) / 2)
    acc = accuracy_score(y_true, y_pred)
    d   = 1 + z**2 / n
    center = (acc + z**2 / (2 * n)) / d
    margin = (z * np.sqrt(acc * (1 - acc) / n + z**2 / (4 * n**2))) / d
    lo, hi = center - margin, center + margin
    print(f"Wilson CI ({conf*100:.0f}%): "
          f"Acc={acc*100:.2f}% [{lo*100:.2f}%, {hi*100:.2f}%]")
    return lo, hi

prop_preds = test_preds['Proposed']['pred']
prop_true  = test_preds['Proposed']['true']

# McNemar & Wilcoxon vs baselines require their CSV files in CV_Results/.
# Uncomment and adjust the list below once baseline results are available:
# BASELINES = ['CNN', 'ResNet50_BL', 'VGG16_BL', 'MobileNetV2_BL', 'InceptionV3_BL']
# print("\n--- McNemar Tests ---")
# for name in BASELINES:
#     if name in test_preds:
#         mcnemar_test(prop_true, prop_preds, test_preds[name]['pred'], name)
# print("\n--- Wilcoxon Signed-Rank Tests (5-Fold) ---")
# for name in BASELINES:
#     wilcoxon_test('Proposed', name, n_folds=5)
# print("\n--- Wilcoxon Signed-Rank Tests (10-Fold) ---")
# for name in BASELINES:
#     wilcoxon_test('Proposed', name, n_folds=10)

print("\n--- Wilson Confidence Interval (Proposed) ---")
wilson_ci(prop_true, prop_preds)

res = {n: {'Acc':  accuracy_score(p['true'], p['pred']) * 100,
           'Prec': precision_score(p['true'], p['pred'], average='weighted') * 100,
           'Rec':  recall_score(p['true'], p['pred'], average='weighted')   * 100,
           'F1':   f1_score(p['true'], p['pred'], average='weighted')       * 100}
      for n, p in test_preds.items()}

test_df = pd.DataFrame(res).T
test_df.to_csv(f'{OUTPUT_DIR}/Test_Results.csv')
print(f"\n--- Test Results ---")
print(test_df.to_string())
print(f"\n All results saved to: {OUTPUT_DIR}/")



  HELD-OUT TEST EVALUATION & STATISTICAL ANALYSIS
  MobileNetV2 backbone: 104 frozen | 35 trainable (last 35)
  Loaded checkpoint: CV_Results/BestModels/Proposed_best.pth

--- Wilson Confidence Interval (Proposed) ---
Wilson CI (95%): Acc=98.24% [97.70%, 98.65%]

--- Test Results ---
                Acc       Prec        Rec         F1
Proposed  98.238482  98.252305  98.238482  98.240396

 All results saved to: CV_Results/
